In [ ]:
!pip install -q litgpt transformers datasets torch huggingface_hub sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 155.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.0 MB/s eta 0:00:00


In [ ]:
# @title 2. Authenticate
# Enter your token to access meta-llama/Llama-3.2-1B
try:
    token = userdata.get('HF_TOKEN')
    login(token=token)
except:
    login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# @title 3. Load & Prepare Alpaca (First 100 Samples)
# Standardizing the data input
print("Loading Alpaca dataset...")
ds = load_dataset("tatsu-lab/alpaca", split="train")
samples = ds.select(range(100))

prompts = []
for item in samples:
    if item['input']:
        prompt = f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n"
    else:
        prompt = f"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\n{item['instruction']}\n\n### Response:\n"
    prompts.append(prompt)

print(f"Prepared {len(prompts)} prompts.")

Loading Alpaca dataset...
Prepared 100 prompts.


In [ ]:
# @title 4. Benchmark: Hugging Face (Sequential - Matched to LitGPT)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

model_id = "meta-llama/Llama-3.2-1B"
print(f"Loading HF Model: {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

hf_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32, # Ensure consistent precision
    device_map="cuda"
)

# Container for captured matrices
hf_matrices = {"Q": [], "K": [], "V": []}

def hf_hook(module, args, output):
    # Output of a linear layer is [Batch, Seq, Hidden]
    # In sequential mode, Batch will be 1
    hf_matrices["Q"].append(output.detach().cpu())

# Hook specifically the Query projection layer
# Llama-3.2 structure: model.layers[i].self_attn.q_proj
layer_index = 0
target_layer = hf_model.model.layers[layer_index].self_attn.q_proj
h_hf = target_layer.register_forward_hook(hf_hook)

print("Starting HF Inference (Sequential)...")
start_time = time.time()

# FIX: Run sequentially to match LitGPT's behavior
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # Generate 1 token
    _ = hf_model.generate(**inputs, max_new_tokens=1)

end_time = time.time()
hf_duration = end_time - start_time

h_hf.remove()
torch.save(hf_matrices, "hf_matrices_unopt.pt")

print(f"HF Time: {hf_duration:.4f} s")
print(f"Captured {len(hf_matrices['Q'])} matrices.")

del hf_model, tokenizer
clear_gpu()

Loading HF Model: meta-llama/Llama-3.2-1B...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Starting HF Inference (Sequential)...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

HF Time: 2.9259 s
Captured 100 matrices.


In [ ]:
# @title 5. Benchmark: LitGPT (Sequential Fix)
import huggingface_hub.constants
from litgpt import LLM
from litgpt.api import LLM as LitLLM
from pathlib import Path
import torch.nn as nn
import time

# --- HOTFIX: Patch missing constant ---
if not hasattr(huggingface_hub.constants, "HF_HUB_ENABLE_HF_TRANSFER"):
    setattr(huggingface_hub.constants, "HF_HUB_ENABLE_HF_TRANSFER", False)

model_id = "meta-llama/Llama-3.2-1B"
layer_index = 0

print(f"Loading LitGPT Model: {model_id}...")

# Robust loading
try:
    llm = LitLLM.load(model_id)
except Exception as e:
    print(f"Auto-load failed: {e}")
    from litgpt.scripts.download import download_from_hub
    from litgpt.scripts.convert_hf import convert_hf

    download_from_hub(repo_id=model_id)
    checkpoint_dir = Path(f"checkpoints/{model_id}")
    convert_hf(checkpoint_dir=checkpoint_dir)
    llm = LitLLM.load(checkpoint_dir)

print("Model loaded successfully.")

# Config
config = llm.model.config
n_head = config.n_head
n_query_groups = config.n_query_groups
head_dim = config.head_size

# Calculate sizes
shape_q = n_head * head_dim
shape_kv = n_query_groups * head_dim

lit_matrices = {"Q": [], "K": [], "V": []}

def lit_hook(module, args, output):
    # LitGPT 'qkv' output is fused: [Batch, Seq, Q + K + V]
    q_end = shape_q
    k_end = shape_q + shape_kv

    q = output[..., :q_end]
    k = output[..., q_end:k_end]
    v = output[..., k_end:]

    lit_matrices["Q"].append(q.detach().cpu())
    lit_matrices["K"].append(k.detach().cpu())
    lit_matrices["V"].append(v.detach().cpu())

# --- DIRECT LAYER TARGET ---
# Llama 3.2 uses 'qkv' linear layer in LitGPT
target_layer = llm.model.transformer.h[layer_index].attn.qkv
print(f"Hooking into layer: {target_layer}")
h_lit = target_layer.register_forward_hook(lit_hook)

print("Starting LitGPT Inference...")
start_time = time.time()

# FIX: Loop sequentially.
# LitGPT's high-level 'generate' API does not support list inputs (batches) natively.
for i, prompt in enumerate(prompts):
    # generate() returns the string, we ignore it to focus on metric extraction
    _ = llm.generate(prompt, max_new_tokens=1)

end_time = time.time()
lit_duration = end_time - start_time

h_lit.remove()
torch.save(lit_matrices, "litgpt_matrices_unopt.pt")

print(f"LitGPT Time: {lit_duration:.4f} s")
print(f"Captured {len(lit_matrices['Q'])} matrices.")

# Cleanup
del llm
clear_gpu()

Loading LitGPT Model: meta-llama/Llama-3.2-1B...
Model loaded successfully.
Hooking into layer: Linear(in_features=2048, out_features=3072, bias=False)
Starting LitGPT Inference...
LitGPT Time: 2.5956 s
Captured 100 matrices.


In [ ]:
# @title 6. Final Comparison & Validation
print("--- Baseline Comparison (No Optimizations) ---")
print(f"HF Time:     {hf_duration:.4f} s")
print(f"LitGPT Time: {lit_duration:.4f} s")

# Verify shapes match
hf_q = torch.load("hf_matrices_unopt.pt")['Q'][0]
lit_q = torch.load("litgpt_matrices_unopt.pt")['Q'][0]

print("\n--- Matrix Validation ---")
print(f"HF Q Shape:     {hf_q.shape}")
print(f"LitGPT Q Shape: {lit_q.shape}")

if hf_q.shape == lit_q.shape:
    print("SUCCESS: Both frameworks extracted identical matrix dimensions.")
else:
    print("WARNING: Dimension mismatch (likely due to padding differences in batching).")

--- Baseline Comparison (No Optimizations) ---
HF Time:     2.9259 s
LitGPT Time: 2.5956 s

--- Matrix Validation ---
HF Q Shape:     torch.Size([1, 32, 2048])
LitGPT Q Shape: torch.Size([1, 32, 2048])
SUCCESS: Both frameworks extracted identical matrix dimensions.
